# Notebook 8 — Evaluation: how long to the alarm, and how long to the diagnosis?

The rig is a **detector followed by a diagnoser**: a CUSUM on the healthy-shadow
innovation raises the alarm, and an MMAE bank of fault hypotheses names the cause. This
notebook runs every fault scenario through it twice — once on the ethanol soft sensor
alone, once with the rig's own T/pH/DO instruments added — and reports two numbers each
time:

* **alarm delay** — minutes from the fault to the CUSUM firing;
* **verdict delay** — minutes from the fault to the MMAE naming the right cause *and
  keeping it* to the end of the batch.

| system | detection | isolation |
|---|---|---|
| **E** | CUSUM on ethanol | MMAE scored on ethanol |
| **E + P** | CUSUM on ethanol **and** T/pH/DO, first to fire wins | MMAE scored on ethanol **and** T/pH/DO |

The probes are available to both stages or to neither — an instrument you have paid for
is available to the whole pipeline. E + P therefore alarms earlier *and* has less batch
in hand when its bank is fired; the two columns are what that trade is worth.

One batch per scenario, fault at **t = 0.5 h**, the onset the app's own
`backend/test_mmae_isolation.py` uses. Part A rebuilds the plant, shadow, detector and
bank so the notebook needs nothing but `numpy`, `pandas` and `matplotlib`; Part D checks
that rebuild against the deployed app when the app is present.

## A.1 · Imports

In [72]:
import copy
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.35,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 120,
})
FIGDIR = Path('figures'); FIGDIR.mkdir(exist_ok=True)

# Two series throughout: the soft sensor alone, and the soft sensor + the rig's probes.
C_ETH, C_PRB = '#2a78d6', '#eb6834'      # validated categorical pair (blue / orange)
C_MUTED = '#52514e'

## A.2 · Parameters

Every value below is the deployed default, and three of them are the reason this
notebook prints different numbers from notebook 7 for identical physics: the batch is
**10 h** not 7, glucose starts at **5.0** g/L not 6.0, and the organism is the
**`robust`** profile (cardinal T 5/30/40, cardinal pH 2.5/4.7/8.0) rather than
`sensitive`. Part H quantifies what each one moves.

In [73]:
# ── A.2 · Parameters ─────────────────────────────────────────────────────────
# One dict, grouped as the app groups them. Every value is the deployed default
# (bioreactor_ui/backend/core.py :: DEFAULTS); nothing here is re-tuned for this
# notebook. The three that differ from notebook 7's own header are called out in
# Part H, because they are the whole reason the two artefacts print different
# numbers for the same physics.
PARAMS = {
    "kinetics": {
        "KG": 0.1, "KE": 0.1, "Ygx": 0.15, "Yge": 0.34, "Yex": 0.43,
        # Respiro-fermentative split (Crabtree). Oxygen does NOT scale the
        # growth rate; it sets the FATE of glucose carbon. A DO fault therefore
        # pushes ethanol UP while a T or pH fault pushes it DOWN, and that sign
        # is what tells them apart on an ethanol-only measurement.
        "YGE_FERM": 0.51, "YGX_OX": 0.25, "YGX_RED": 0.10,
        "C_RESP": 0.2573, "YO_E_REL": 2.0, "K_O2": 9.050,
    },
    "initial": {"X0": 2.5, "G0": 5.0, "E0": 0.0, "muG0": 0.15, "muE0": 0.08},
    "thermal": {
        "V": 1.35, "rhoCp": 4180.0, "Y_QX": 12000.0,
        "T_set": 30.0, "T_amb": 22.0, "UA": 18.0, "K_p": 50.0, "Q_max": 630.0,
        "T_min": 5.0, "T_opt": 30.0, "T_max": 40.0,      # 'robust' cardinals
    },
    "ph": {
        "pH_set": 5.0, "pH_min": 2.5, "pH_opt": 4.7, "pH_max": 8.0,
        "alpha_meta": 0.4, "alpha_pump": 0.2563,
        "F_A": 210.0, "F_B": 210.0,
        "Kc_pump": 120.0, "Ki_pump": 250.0, "DT_CTRL_s": 2.0,
    },
    "oxygen": {
        "N_rpm": 500.0, "Q_air_Lmin": 3.5, "DO_star": 100.0, "DO0": 100.0,
        "qO2_max": 8.0, "K_O": 3.0, "DO_sat_mgL": 6.7,
    },
    "ekf": {
        "sigma_E": 0.3162,        # what the FILTER assumes -> R = 0.1
        "sigma_E_sensor": 0.05,   # what the SENSOR actually does
        "cadence_min": 5.0, "cusum_k": 0.15, "cusum_h": 1.5,
        "sigma_T_probe": 0.1, "sigma_pH_probe": 0.02, "sigma_DO_probe": 1.0,
    },
    "sim": {"t_final": 10.0, "dt_min": 1.0},
}

# Quoted by the UI as a release gate on the root-cause panel; measured, not used,
# in this notebook. Part E is what measures them.
ETHANOL_SETTLE_MIN, PROBE_SETTLE_MIN = 80.0, 30.0
K_SLACK, H_CUSUM = PARAMS["ekf"]["cusum_k"], PARAMS["ekf"]["cusum_h"]

N_STATE = 8
IDX = {"T": 5, "pH": 6, "DO": 7}      # positions of T, pH, DO in the 8-state vector

print(f"batch {PARAMS['sim']['t_final']} h at {PARAMS['sim']['dt_min']} min/frame, "
      f"ethanol every {PARAMS['ekf']['cadence_min']} min")
print(f"deployed CUSUM  : k = {K_SLACK}  h = {H_CUSUM}   (units of sqrt(S))")
print(f"quoted constants: ETHANOL_SETTLE_MIN = {ETHANOL_SETTLE_MIN}"
      f"  PROBE_SETTLE_MIN = {PROBE_SETTLE_MIN}")

batch 10.0 h at 1.0 min/frame, ethanol every 5.0 min
deployed CUSUM  : k = 0.15  h = 1.5   (units of sqrt(S))
quoted constants: ETHANOL_SETTLE_MIN = 80.0  PROBE_SETTLE_MIN = 30.0


## A.3 · Growth modifiers, the carbon split, and kLa from the vessel

Identical to notebook 7 §A.3 and notebook 6 §2. Two things in here are load-bearing and
were both bugs once:

* **`f_pH` uses the Rosso n = 1 form.** The n = 2 CTMI has a root in its denominator
  whenever the optimum sits below the window midpoint, and the clip to [0, 1] hides that
  pole as a clean **step function** — it can say "dead" or "fine" and nothing between.
  `_assert_monotone` is the regression test.
* **Oxygen does not scale the growth rate.** It sets the *fate* of glucose carbon, so a
  DO fault pushes ethanol **up** while a T or pH fault pushes it **down**. That sign is
  the only thing separating the oxygen loop from the others on an ethanol-only sensor.

In [74]:
# ── A.3 · Growth modifiers, the carbon split, and kLa from the vessel ────────
PHI_FLOOR = 1e-6      # never return a hard zero: that hands the EKF a zero
                      # Jacobian and the filter can never correct its way back.


def _rosso_n2(v, v_min, v_opt, v_max):
    """Rosso (1993) CTMI, n = 2 — the cardinal TEMPERATURE model."""
    if v <= v_min or v >= v_max:
        return 0.0
    num = (v - v_max) * (v - v_min) ** 2
    den = (v_opt - v_min) * ((v_opt - v_min) * (v - v_opt)
                             - (v_opt - v_max) * (v_opt + v_min - 2 * v))
    return 0.0 if abs(den) < 1e-12 else num / den


def _rosso_n1(v, v_min, v_opt, v_max):
    """Rosso n = 1 — the pH form. No interior pole, asymmetric arms handled."""
    if v <= v_min or v >= v_max:
        return 0.0
    num = (v - v_min) * (v - v_max)
    den = num - (v - v_opt) ** 2
    return 0.0 if abs(den) < 1e-12 else num / den


def make_cardinal_modifier(v_min, v_opt, v_max, anchor, n=2):
    """Re-anchored Rosso modifier with f(anchor) == 1.

    Clipping happens AFTER re-anchoring and only from below: clipping to [0, 1]
    first would truncate the physically correct f > 1 that arises when the
    reactor sits nearer the optimum than the anchor does.

    `n` is about the SHAPE OF THE WINDOW, not about which variable it models.
    n = 2 is valid only while the optimum sits at or above the window midpoint;
    n = 1 is pole-free for any arms. A pole inside the window is invisible in the
    output — the clip presents it as a clean 0/1 step — which is exactly how
    f_pH once became a step function and made a stuck acid pump look like an
    oxygen crash. `_assert_monotone` below is the regression test.
    """
    raw = _rosso_n1 if n == 1 else _rosso_n2
    a = raw(anchor, v_min, v_opt, v_max)
    if a < 1e-9:
        raise ValueError(
            f"anchor {anchor} lies outside the viable window ({v_min}, {v_max}): "
            f"the modifier is normalised at the setpoint, so a setpoint the "
            f"organism cannot grow at leaves the HEALTHY reference dead.")
    return lambda v: max(raw(v, v_min, v_opt, v_max) / a, PHI_FLOOR)


def make_do_modifier(K_O, DO_star):
    """Re-anchored Monod oxygen limitation, f_DO(DO_star) == 1."""
    anchor = DO_star / (K_O + DO_star)

    def f_DO(DO):
        DO = max(DO, 0.0)
        return max(0.0, min(1.0, (DO / (K_O + DO)) / anchor))
    return f_DO


def absolute_modifier(v, v_min, v_opt, v_max, n=2):
    """The modifier relative to the organism's TRUE optimum, un-anchored.

    `make_cardinal_modifier` normalises at the SETPOINT, so f(setpoint) is
    identically 1 — useful for "how far from where we hold it", useless for "is
    where we hold it any good". This answers the second question.
    """
    raw = _rosso_n1 if n == 1 else _rosso_n2
    peak = raw(v_opt, v_min, v_opt, v_max)
    return 0.0 if peak < 1e-12 else raw(v, v_min, v_opt, v_max) / peak


def _assert_monotone(f, lo, hi, name, n=300):
    """Every modifier must rise monotonically towards its optimum."""
    step = (hi - lo) / (n - 1)
    prev = f(lo)
    for i in range(1, n):
        cur = f(lo + i * step)
        if cur - prev < -1e-9:
            raise ValueError(f"{name} is not monotone rising on [{lo}, {hi}]: a "
                             f"Rosso denominator has changed sign and the clip "
                             f"is hiding the pole as a step function.")
        prev = cur


def build_modifiers(p):
    """(f_T, f_pH, f_DO) for the current cardinals, with the shape guards."""
    th, ph, ox = p["thermal"], p["ph"], p["oxygen"]
    n_T = 2 if (th["T_opt"] - th["T_min"]) >= (th["T_max"] - th["T_opt"]) else 1
    f_T = make_cardinal_modifier(th["T_min"], th["T_opt"], th["T_max"],
                                 anchor=th["T_set"], n=n_T)
    # pH arms are asymmetric -> n = 1, anchored at the SETPOINT.
    f_pH = make_cardinal_modifier(ph["pH_min"], ph["pH_opt"], ph["pH_max"],
                                  anchor=ph["pH_set"], n=1)
    f_DO = make_do_modifier(ox["K_O"], ox["DO_star"])
    _assert_monotone(f_T, th["T_min"] + 1e-6, th["T_opt"], "f_T")
    _assert_monotone(f_pH, ph["pH_min"] + 1e-6, ph["pH_opt"], "f_pH")
    _assert_monotone(f_DO, 0.0, ox["DO_star"], "f_DO")
    return f_T, f_pH, f_DO


def carbon_fluxes(G, E, mu_max_G, mu_max_E, kin, phi_g=1.0, f_do=1.0):
    """Split glucose uptake between respiration and fermentation.

    Returns (q_ox, q_fm, mu_E, mu) per g X per h. The respiratory capacity is
    scaled by phi_g too: respiration is enzymatic, so cold or acid slows it
    exactly as it slows uptake. Leaving the cap at its 30 degC value made a
    chilled culture MORE respiratory — fermentation stopped, and ethanol, the
    only measurement, went silent on the very fault it had to see.
    """
    KG, KE, Ygx = kin["KG"], kin["KE"], kin["Ygx"]
    if mu_max_G > 1e-9:
        q_G = (mu_max_G * G / (KG + G) * phi_g) / Ygx
        suppression = 1.0 - (q_G * Ygx) / mu_max_G          # diauxic repression
    else:
        q_G, suppression = 0.0, 1.0
    q_ox = min(q_G, kin["C_RESP"] * phi_g * f_do)
    q_fm = q_G - q_ox                                       # overflow -> fermentation
    # ethanol is a RESPIRATORY substrate: no oxygen, no consumption
    mu_E = (mu_max_E * E / (KE + E) * suppression * phi_g * f_do
            ) if mu_max_E > 1e-9 else 0.0
    mu = q_ox * kin["YGX_OX"] + q_fm * kin["YGX_RED"] + mu_E
    return q_ox, q_fm, mu_E, mu


# ── kLa COMPUTED FROM THE VESSEL, not assumed ────────────────────────────────
# Van't Riet on the Minifors 2 geometry: kLa = C·(Pg/V)^a·v_s^b. Comes out at
# 44.8 1/h. Same function in 4_fault_injection, 7_mmae_fault_isolation and
# bioreactor_ui/backend/plant.py, so all four artefacts run one number.
_D_VESSEL_M, _D_IMP_FRAC, _N_POWER, _N_IMPELLERS = 0.090, 1 / 3, 5.0, 2
_RHO_BROTH, _PG_P0 = 1000.0, 0.5
_VR_C, _VR_A, _VR_B = 0.002, 0.7, 0.2


def compute_kLa(N_rpm, Q_air_Lmin, V_L):
    """Volumetric O2 transfer coefficient [1/h] from agitation + aeration."""
    A_cross = np.pi / 4.0 * _D_VESSEL_M ** 2
    D_imp = _D_VESSEL_M * _D_IMP_FRAC
    v_s = (Q_air_Lmin / 1000.0 / 60.0) / A_cross            # superficial gas vel [m/s]
    P0 = _N_IMPELLERS * _N_POWER * _RHO_BROTH * (N_rpm / 60.0) ** 3 * D_imp ** 5
    Pg_V = (_PG_P0 * P0) / (V_L / 1000.0)
    if Pg_V <= 0.0 or v_s <= 0.0:
        return 0.0
    return _VR_C * (Pg_V ** _VR_A) * (v_s ** _VR_B) * 3600.0


_fT, _fpH, _fDO = build_modifiers(PARAMS)
_kLa0 = compute_kLa(PARAMS["oxygen"]["N_rpm"], PARAMS["oxygen"]["Q_air_Lmin"],
                    PARAMS["thermal"]["V"])
_th = PARAMS["thermal"]
T_SS = ((_th["K_p"] * _th["T_set"] + _th["UA"] * _th["T_amb"])
        / (_th["K_p"] + _th["UA"]))
print(f"kLa (healthy, from geometry) = {_kLa0:.1f} 1/h")
print(f"thermal steady state         = {T_SS:.2f} degC   (NOT T_set: a P heater "
      f"needs a standing error)   f_T there = {_fT(T_SS):.3f}")
print(f"f_pH at the setpoint, absolute = "
      f"{absolute_modifier(5.0, 2.5, 4.7, 8.0, n=1):.3f}")

kLa (healthy, from geometry) = 44.8 1/h
thermal steady state         = 27.88 degC   (NOT T_set: a P heater needs a standing error)   f_T there = 0.974
f_pH at the setpoint, absolute = 0.988


## A.4 · Faults, and A.5 · the 8-state plant

A fault never edits a state. It changes the **actuator context** of a control tick —
pump flows, heater fraction, kLa — and the ODE does the rest. That is what makes a fault
something the filter can be *wrong about* in a way the data can score.

In [75]:
# ── A.4 · The fault registry, and how a fault reaches the plant ──────────────
# Four actuator loops x two failure modes. A fault never edits a state: it
# changes the ACTUATOR CONTEXT of a control tick, and the ODE does the rest.
#
# AGITATION AND AERATION ARE ONE MECHANISM. Both reach the biology only through
# kLa, so any (rpm, air-flow) pair giving the same kLa produces a bit-identical
# trajectory in all 8 states — provably equal to 1e-9. They are one hypothesis
# parameterised two ways, and only an instrument on the actuator itself (a
# tachometer, an air mass-flow meter) could separate them. Severity is therefore
# a kLa RATIO, not an rpm.
FAULT_DEFAULTS = {
    "heater_off":         {"Q_pct": 0.0},        # % of heater power still delivered
    "heater_clog":        {"Q_pct": 6.0},
    "acid_pump_off":      {},                    # F_A forced to 0 (dead)
    "acid_pump_stuck":    {"F_A_stuck": 210.0},  # mL/h = pump saturation
    "base_pump_off":      {},                    # F_B forced to 0 (dead)
    "base_pump_stuck":    {"F_B_stuck": 210.0},
    "agitation_aer_stop": {"kla_pct": 0.0},      # % of healthy kLa still delivered
    "agitation_aer_clog": {"kla_pct": 3.0},
}


def normalize_fault(fault):
    """Fill a fault request with the defaults for its type."""
    fid = fault.get("fault") or fault.get("id")
    merged = dict(FAULT_DEFAULTS.get(fid, {}))
    merged.update({k: v for k, v in fault.items() if k not in ("fault", "id")})
    merged["fault"] = fid
    return merged


def apply_faults(active, t, p, base_F_A, base_F_B):
    """Resolve the actuator context at time t. Returns (F_A, F_B, heater_frac,
    UA_eff, kLa_eff)."""
    ox = p["oxygen"]
    F_A, F_B = base_F_A, base_F_B
    heater_frac, UA_eff = 1.0, p["thermal"]["UA"]
    kLa_eff = compute_kLa(ox["N_rpm"], ox["Q_air_Lmin"], p["thermal"]["V"])
    for f in active:
        if t < f.get("t_fault", 0.0):
            continue
        fid = f["fault"]
        if fid in ("heater_off", "heater_clog"):
            heater_frac = min(max(f.get("Q_pct", 0.0), 0.0), 100.0) / 100.0
        elif fid == "acid_pump_off":
            F_A = 0.0
        elif fid == "acid_pump_stuck":
            F_A = f.get("F_A_stuck", 210.0)
        elif fid == "base_pump_off":
            F_B = 0.0
        elif fid == "base_pump_stuck":
            F_B = f.get("F_B_stuck", 210.0)
        elif fid in ("agitation_aer_stop", "agitation_aer_clog"):
            kLa_eff *= min(max(f.get("kla_pct", 0.0), 0.0), 100.0) / 100.0
    return F_A, F_B, heater_frac, UA_eff, kLa_eff


# ── A.5 · The 8-state plant ──────────────────────────────────────────────────
def make_rhs(p, f_T, f_pH, f_DO, F_A, F_B, heater_frac, UA_eff, kLa_eff):
    """The 8-state RHS closure for ONE control tick, with the actuator context
    held fixed over it — which is what makes the discrete pH controller discrete.
    """
    kin, th, ph, ox = p["kinetics"], p["thermal"], p["ph"], p["oxygen"]
    Yex = kin["Yex"]
    V, rhoCp, Y_QX = th["V"], th["rhoCp"], th["Y_QX"]
    T_set, T_amb, K_p, Q_max = th["T_set"], th["T_amb"], th["K_p"], th["Q_max"]
    alpha_meta, alpha_pump = ph["alpha_meta"], ph["alpha_pump"]
    DO_star, DO_sat_mgL = ox["DO_star"], ox["DO_sat_mgL"]

    def rhs(t, y):
        X, G, E, m1, m2, T, pH, DO = y
        X = max(X, 0.0); G = max(G, 0.0); E = max(E, 0.0)
        pH = max(pH, 1.0); DO = max(DO, 0.0)

        phi_g = f_T(T) * f_pH(pH)          # RATE: temperature and pH only
        f_do = f_DO(DO)                    # FATE: oxygen caps respiration
        q_ox, q_fm, muE, mu = carbon_fluxes(G, E, m1, m2, kin, phi_g, f_do)
        q_Eox = muE / Yex                  # ethanol consumed [g E / g X / h]

        dX = mu * X
        dG = -(q_ox + q_fm) * X
        dE = (q_fm * kin["YGE_FERM"] - q_Eox) * X

        # Thermal ODE (W -> J/h via x3600; metabolic heat already J/h)
        Q_heat = heater_frac * max(0.0, min(Q_max, K_p * (T_set - T)))
        dT = (3600.0 * (Q_heat - UA_eff * (T - T_amb)) + Y_QX * mu * X * V) / (V * rhoCp)

        # pH ODE: yeast acidogenesis against the pump flows held for this tick
        dpH = -alpha_meta * mu * X + alpha_pump * F_B - alpha_pump * F_A
        if pH <= 1.0 and dpH < 0:
            dpH = 0.0

        # DO ODE (%-sat/h): OTR supply - respiratory OUR demand. Demand tracks
        # the OXIDATIVE flux only, so the fermentative branch draws no oxygen.
        demand = kin["K_O2"] * (q_ox + kin["YO_E_REL"] * q_Eox) * X
        dDO = kLa_eff * (DO_star - DO) - demand / DO_sat_mgL * 100.0
        if DO <= 0.0 and dDO < 0:
            dDO = 0.0
        return [dX, dG, dE, 0.0, 0.0, dT, dpH, dDO]
    return rhs


def _rk4(p, mods, y, t, dt, ctx):
    """One RK4 step with the actuator context `ctx` fixed."""
    rhs = make_rhs(p, *mods, *ctx)
    k1 = np.array(rhs(t, y))
    k2 = np.array(rhs(t + dt / 2, y + dt / 2 * k1))
    k3 = np.array(rhs(t + dt / 2, y + dt / 2 * k2))
    k4 = np.array(rhs(t + dt, y + dt * k3))
    y = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    y[0] = max(y[0], 0.0); y[1] = max(y[1], 0.0)
    y[2] = max(y[2], 0.0); y[7] = max(y[7], 0.0)
    return y


def advance(p, mods, y, t0, t1, faults, integ):
    """Integrate [t0, t1) with PI pH control on a short tick.

    The integral term is what slowly pushes pH back to the setpoint after a
    fault dips it; a pure-P law leaves a standing offset. Returns (y, integ) —
    the integral is carried per TRAJECTORY, so the healthy counterfactual and
    every MMAE hypothesis control themselves independently.
    """
    ph = p["ph"]
    pH_set, Kc, Ki = ph["pH_set"], ph["Kc_pump"], ph["Ki_pump"]
    F_A_max, F_B_max = ph["F_A"], ph["F_B"]
    dt_ctrl = max(ph["DT_CTRL_s"], 0.5) / 3600.0
    y = np.array(y, dtype=float)
    t = t0
    while t < t1 - 1e-12:
        h = min(dt_ctrl, t1 - t)
        err = pH_set - y[6]                       # >0 -> too acidic, add base
        u = Kc * err + Ki * integ                 # signed pump command [mL/h]
        if u >= 0.0:
            base_F_B, base_F_A = min(u, F_B_max), 0.0
        else:
            base_F_A, base_F_B = min(-u, F_A_max), 0.0
        # Anti-windup: do not accumulate while saturated and driving deeper in.
        saturated = u > F_B_max or u < -F_A_max
        if not (saturated and (err > 0.0) == (u > 0.0)):
            integ += err * h
        ctx = apply_faults(faults, t, p, base_F_A, base_F_B)
        y = _rk4(p, mods, y, t, h, ctx)
        t += h
    return y, integ


def rhs_for(p, mods, y, integ, faults, t):
    """The RHS in the actuator context state `y` implies at `t`.

    Same pump law and same fault resolution as `advance`, so the Jacobian is
    taken of exactly the dynamics the filter is propagated with. `faults=[]`
    gives the healthy shadow; an MMAE hypothesis passes its own list.
    """
    ph = p["ph"]
    u = ph["Kc_pump"] * (ph["pH_set"] - y[6]) + ph["Ki_pump"] * integ
    if u >= 0.0:
        F_B, F_A = min(u, ph["F_B"]), 0.0
    else:
        F_A, F_B = min(-u, ph["F_A"]), 0.0
    return make_rhs(p, *mods, *apply_faults(faults, t, p, F_A, F_B))

## A.6 · The healthy shadow and the CUSUM, and A.7 · one run of the rig

The filter integrates the **healthy** ODEs and is corrected by ethanol alone, so it
carries a healthy *expectation* of the environment. A filter fed the measured T/pH/DO
instead would slow its prediction by exactly as much as reality and stay blind — that is
Design B, and notebook 6 measures it going blind on a fault this one alarms on.

In [76]:
# ── A.6 · The Design-A healthy shadow and the CUSUM ──────────────────────────
# T, pH and DO are ESTIMATED states here, integrated from the HEALTHY ODEs and
# never fed in. That is the whole difference from Design B, where they are
# measured inputs folded into phi and the filter therefore FOLLOWS every fault.
# Because this filter only integrates healthy dynamics and is corrected by
# ethanol alone, it answers "how should this reactor be behaving?" — and the
# innovation is the fault signal.
P0 = np.diag([0.1, 0.02, 0.02, 1e-5, 1e-5, 1e-4, 1e-6, 1e-4])
Q8 = np.diag([1e-3, 1e-3, 1e-3, 1e-9, 1e-9, 1e-4, 1e-6, 1e-4])
Hm = np.zeros((1, N_STATE)); Hm[0, 2] = 1.0          # ethanol-only observation
# mumax is PINNED (1e-5 / 1e-9): the filter must not be able to absorb a fault
# by retuning the biology. A rigid reference is what keeps the fault visible.


def compute_jacobian(rhs, x, eps=1e-6):
    """Forward-difference Jacobian (8x8) of the healthy dynamics at x.

    Forward differences, not SymPy: the carbon split contains
    min(q_G, C_RESP*phi_g*f_do) and is not differentiable at the
    respiro-fermentative switch — exactly where a DO fault lives.
    """
    f0 = np.array(rhs(0.0, x), dtype=float)
    J = np.zeros((N_STATE, N_STATE))
    for j in range(N_STATE):
        xp = np.array(x, dtype=float)
        xp[j] += eps
        J[:, j] = (np.array(rhs(0.0, xp), dtype=float) - f0) / eps
    return J


class EKF8:
    """Continuous-discrete healthy-shadow EKF, stepped one frame at a time.

    State prediction does not live here: the caller advances this filter through
    the very same integrator that produces the healthy counterfactual, so "the
    EKF predicts with healthy dynamics" is true by construction rather than by a
    second copy of the ODEs that could drift out of sync.
    """

    def __init__(self, p):
        ini, th, ph, ox = p["initial"], p["thermal"], p["ph"], p["oxygen"]
        self.x = np.array([ini["X0"], ini["G0"], ini["E0"], ini["muG0"],
                           ini["muE0"], th["T_set"], ph["pH_set"], ox["DO0"]],
                          dtype=float)
        self.P = P0.copy()
        self.Q = Q8.copy()
        self.R = np.array([[max(p["ekf"]["sigma_E"] ** 2, 1e-4)]])
        self.I8 = np.eye(N_STATE)
        self.pi_int = 0.0            # the shadow runs its own pH controller
        self.prev_x = self.x.copy()

    def predict_cov(self, dt_h, rhs):
        """P <- Phi P Phi' + Q dt, with Phi = I + F dt at the current state."""
        Phi = self.I8 + compute_jacobian(rhs, self.x) * dt_h
        self.P = Phi @ self.P @ Phi.T + self.Q * dt_h

    def correct(self, z_ethanol):
        """Joseph-form update on ethanol. Returns (innovation, S)."""
        S = Hm @ self.P @ Hm.T + self.R
        K = self.P @ Hm.T @ np.linalg.inv(S)
        nu = float(z_ethanol - self.x[2])
        self.x = self.x + (K @ np.array([[nu]])).flatten()
        for i in (0, 1, 2, 7):
            self.x[i] = max(self.x[i], 0.0)
        self.x[6] = max(self.x[6], 1.0)
        # Batch mass balance, enforced on the CORRECTION. The plant has no death
        # term and no feed, so a correction that reverses either is inadmissible.
        # It matters because mumax is pinned: without this the filter absorbs the
        # fault into X and G instead, dragging X below the faulted truth and
        # CREATING glucose.
        self.x[0] = max(self.x[0], self.prev_x[0])
        self.x[1] = min(self.x[1], self.prev_x[1])
        IKH = self.I8 - K @ Hm
        self.P = IKH @ self.P @ IKH.T + K @ self.R @ K.T
        self.prev_x = self.x.copy()
        return nu, float(S[0, 0])


def cusum_step(sp, sn, z, k=K_SLACK, h=H_CUSUM):
    """ONE CUSUM update — the detector as it runs on the rig.

        S+ <- max(0, S+ + z - k)     ethanol ABOVE prediction
        S- <- max(0, S- - z - k)     ethanol BELOW prediction  <- fault direction

    `z` is the innovation normalised by the filter's own predicted spread
    sqrt(S), so a fixed threshold means the same thing at every step. Two floats
    of state, no history. The arms are NOT reset after firing: this study asks
    when a fault was FIRST seen, so they stay a monotone evidence trace.
    """
    sp = max(0.0, sp + z - k)
    sn = max(0.0, sn - z - k)
    return sp, sn, (sp > h or sn > h)


# ── A.7 · One run of the rig ─────────────────────────────────────────────────
class Simulation:
    """The plant, the healthy shadow and the detector, advanced frame by frame.

    Frame = 1 sim-minute, and each frame carries several PI control ticks. The
    same `advance` call produces the faulted trajectory, the no-fault
    counterfactual and the shadow's prediction — the only difference is the
    fault list handed to it.
    """

    def __init__(self, cfg=None):
        self.params = copy.deepcopy(PARAMS)
        for group, vals in (cfg or {}).items():
            self.params[group].update(vals)
        self.mods = build_modifiers(self.params)

        p = self.params
        ini, th, ph, ox = p["initial"], p["thermal"], p["ph"], p["oxygen"]
        self.y = np.array([ini["X0"], ini["G0"], ini["E0"], ini["muG0"],
                           ini["muE0"], th["T_set"], ph["pH_set"], ox["DO0"]],
                          dtype=float)
        self.healthy_y = self.y.copy()
        self.t, self.t_final = 0.0, p["sim"]["t_final"]
        self.dt_h = p["sim"]["dt_min"] / 60.0
        self.active_faults = []
        self.pi_int = self.pi_int_healthy = 0.0

        self.ekf = EKF8(p)
        self.sigma_E = p["ekf"]["sigma_E_sensor"]      # the REAL sensor noise
        self.cadence_h = p["ekf"]["cadence_min"] / 60.0
        self.next_meas_t = self.cadence_h
        self.rng = np.random.default_rng(0)

        self.cusum_k, self.cusum_h = p["ekf"]["cusum_k"], p["ekf"]["cusum_h"]
        self.sp = self.sn = 0.0
        self.alarm_t, self.alarm_idx = None, None
        self.rows, self.meas_rows, self.healthy_rows = [], [], []

    def inject_fault(self, fault):
        f = normalize_fault(fault)
        f["t_fault"] = float(f.get("t_fault", self.t))
        if f["t_fault"] < self.t - 1e-9:
            return False, "t_fault has already passed."
        self.active_faults.append(f)
        return True, f"{f['fault']} scheduled at t = {f['t_fault']:.2f} h."

    @property
    def done(self):
        return self.t >= self.t_final - 1e-9

    def advance(self, y, t0, t1, faults, integ):
        return advance(self.params, self.mods, y, t0, t1, faults, integ)

    def rhs_for(self, y, integ, faults, t):
        return rhs_for(self.params, self.mods, y, integ, faults, t)

    def step(self):
        t0, t1 = self.t, min(self.t + self.dt_h, self.t_final)
        self.y, self.pi_int = self.advance(self.y, t0, t1,
                                           self.active_faults, self.pi_int)
        self.healthy_y, self.pi_int_healthy = self.advance(
            self.healthy_y, t0, t1, [], self.pi_int_healthy)
        self.t = t1

        # Design A: predict with the HEALTHY dynamics — the same call that
        # produced the counterfactual above. Nothing about the faulted run enters
        # here; the ethanol correction below is the filter's only contact with
        # reality.
        e = self.ekf
        e.x, e.pi_int = self.advance(e.x, t0, t1, [], e.pi_int)
        e.predict_cov(t1 - t0, self.rhs_for(e.x, e.pi_int, [], t1))

        if self.t >= self.next_meas_t - 1e-9:
            z = self.y[2] + self.rng.normal(0.0, self.sigma_E)
            nu, S = e.correct(z)
            self.next_meas_t += self.cadence_h
            z_norm = nu / max(S ** 0.5, 1e-12)
            self.sp, self.sn, fired = cusum_step(self.sp, self.sn, z_norm,
                                                 self.cusum_k, self.cusum_h)
            self.meas_rows.append({"time_h": self.t, "frame": len(self.rows),
                                   "z": float(z), "nu": nu, "S": S,
                                   "Sp": self.sp, "Sn": self.sn})
            if fired and self.alarm_t is None:
                self.alarm_t = self.t
                self.alarm_idx = len(self.meas_rows) - 1

        X, G, E, _, _, T, pHv, DOv = self.y
        self.rows.append({"time_h": self.t, "T": float(T), "pH": float(pHv),
                          "DO": float(DOv), "G": float(G), "E": float(E),
                          "X": float(X)})
        hX, _, hE, _, _, hT, hpH, hDO = self.healthy_y
        self.healthy_rows.append({"time_h": self.t, "X": float(hX),
                                  "E": float(hE), "T": float(hT),
                                  "pH": float(hpH), "DO": float(hDO)})
        return self.rows[-1]

    def run(self):
        while not self.done:
            self.step()
        return self

In [77]:
# ── A.8 · The MMAE bank: which fault, given the alarm ────────────────────────
# One EKF per candidate cause, every one fed the SAME recorded measurements, each
# believing its own fault started at its own onset. The cause whose filter is
# least surprised by the data wins.
HYPOTHESIS_SEVERITY = {
    "heater_off":         {"Q_pct": 0.0},
    "heater_clog":        {"Q_pct": 6.0},
    "acid_pump_off":      {},
    "acid_pump_stuck":    {"F_A_stuck": 210.0},
    "base_pump_off":      {},
    "base_pump_stuck":    {"F_B_stuck": 210.0},
    "agitation_aer_stop": {"kla_pct": 0.0},
    "agitation_aer_clog": {"kla_pct": 3.0},
}
HYPOTHESIS_LABELS = {
    "heater_off": "Heater dead", "heater_clog": "Heater underpowered",
    "acid_pump_off": "Acid pump dead", "acid_pump_stuck": "Acid pump stuck ON",
    "base_pump_off": "Base pump dead", "base_pump_stuck": "Base pump stuck ON",
    "agitation_aer_stop": "Agitation/aeration stopped",
    "agitation_aer_clog": "Agitation/aeration restricted",
}
NULL = "healthy"      # the null hypothesis: nothing is wrong

# Onsets tried per cause, as offsets [h] from the CUSUM estimate. BACKWARDS ONLY,
# and that is the measured asymmetry: guessing early is nearly free (before its
# own onset a hypothesis simply IS the healthy model), guessing late is fatal.
#
# DEPTH GOES TO -120 min, NOT -60. The oxygen-loop faults alarm late — their
# early innovations sit under the CUSUM slack k and keep resetting the arm, so
# the onset estimate lands up to +105 min after the truth. With a -60 floor the
# ladder cannot reach the real onset, and agitation_aer_clog is then read as
# agitation_aer_stop with p = 1.000 — a confident error produced entirely by the
# floor. (7_mmae_fault_isolation.ipynb, C.2)
LADDER_H = (0.0, -0.5, -1.0, -1.5, -2.0)

# 2.3 nats is 10:1 odds. Two causes closer than this are not distinguishable by
# this data, and an honest tie beats a confident error: the verdict reports every
# cause inside the margin, not just the argmax.
TIE_MARGIN_NATS = 2.3
DEGENERATE_NATS = 1e-6      # below this the two produced IDENTICAL innovations


def candidate_causes():
    return {fid: {"label": HYPOTHESIS_LABELS[fid],
                  "params": {**FAULT_DEFAULTS.get(fid, {}), **sev}}
            for fid, sev in HYPOTHESIS_SEVERITY.items()}


def onset_estimate(meas_rows, alarm_idx):
    """Last time the FIRING arm was zero before the alarm.

    Unreliable in BOTH directions, which is exactly why the ladder exists: on
    healthy data an arm leaves zero ~11x per batch on noise alone, so the walk
    back can run through a pre-fault excursion and land early; and a slow fault
    whose early innovations sit below the slack k keeps resetting the arm, so it
    lands late.
    """
    i = alarm_idx
    key = "Sn" if meas_rows[i]["Sn"] >= meas_rows[i]["Sp"] else "Sp"
    j = i
    while j > 0 and meas_rows[j - 1][key] > 0.0:
        j -= 1
    return float(meas_rows[j]["time_h"])


class _Hyp:
    """A hypothesis filter: the shadow EKF plus the fault list it believes in."""

    def __init__(self, sim, faults):
        self.ekf, self.faults = EKF8(sim.params), faults

    def snapshot(self):
        e = self.ekf
        return {"x": e.x.copy(), "P": e.P.copy(),
                "prev_x": e.prev_x.copy(), "pi_int": e.pi_int}

    def restore(self, s):
        e = self.ekf
        e.x, e.P = s["x"].copy(), s["P"].copy()
        e.prev_x, e.pi_int = s["prev_x"].copy(), s["pi_int"]


def _new_trace(n):
    return {k: np.zeros(n) for k in ("nu", "E", "T", "pH", "DO")}


def _replay(sim, hyp, grid, i0, i1, meas_by_frame, ll, tr, probes, snap_at=None):
    """Advance `hyp` over frames [i0, i1).

    This is the Design-A step with the hypothesis' fault list in place of the
    empty one, and the RECORDED ethanol sample in place of a fresh draw. Both
    scores accumulate in the SAME pass, because they differ only in the
    likelihood, not the trajectory: ll["E"] is the ethanol-only term, ll["P"] the
    extra term contributed by the rig's T/pH/DO instruments. Scoring both costs
    one replay, so the soft-sensor answer and the instrumented answer come out
    side by side instead of forcing a choice made blind.
    """
    e = hyp.ekf
    for i in range(i0, i1):
        if snap_at is not None and i in snap_at:
            snap_at[i].update(hyp.snapshot())
        t0 = grid[i - 1] if i > 0 else 0.0
        t1 = grid[i]
        e.x, e.pi_int = sim.advance(e.x, t0, t1, hyp.faults, e.pi_int)
        e.predict_cov(t1 - t0, sim.rhs_for(e.x, e.pi_int, hyp.faults, t1))
        m = meas_by_frame.get(i)
        if m is not None:
            k = m["k"]
            tr["E"][k] = e.x[2]
            tr["T"][k], tr["pH"][k], tr["DO"][k] = e.x[5], e.x[6], e.x[7]
            nu, S = e.correct(m["z"])
            tr["nu"][k] = nu
            ll["E"][k] = -0.5 * nu * nu / S - 0.5 * math.log(2.0 * math.pi * S)
            # Where the causes actually differ: a dead heater and a jammed acid
            # pump end in the same stalled ethanol curve, but at 22 degC vs
            # 28 degC. This is what breaks the ethanol-only tie.
            acc = 0.0
            for ch in ("T", "pH", "DO"):
                s = probes["sigma"][ch]
                d = probes[ch][k] - tr[ch][k]
                acc += -0.5 * d * d / (s * s) - 0.5 * math.log(2.0 * math.pi * s * s)
            ll["P"][k] = acc
    if snap_at is not None and i1 in snap_at:
        snap_at[i1].update(hyp.snapshot())


def _score(names, runs, key):
    """Collapse one likelihood channel into a verdict: posterior, ties, margin."""
    totals = np.array([runs[c][key]["total"] for c in names])
    order = [int(j) for j in np.argsort(totals)[::-1]]
    p = np.exp(totals - totals.max()); p /= p.sum()      # uniform prior
    margin = float(totals[order[0]] - totals[order[1]]) if len(order) > 1 else None
    tied = [names[j] for j in order
            if totals[order[0]] - totals[j] < TIE_MARGIN_NATS]
    # Two hypotheses with numerically identical scores are degenerate ON THIS
    # DATA: the plant walked the same trajectory under both, so no estimator and
    # no threshold can separate them. The registry no longer contains such a pair
    # — that is why agitation and aeration are one entry — but the check stays,
    # because it is what proved it and would catch the next one.
    degenerate = [[a, b] for i, a in enumerate(names) for b in names[i + 1:]
                  if abs(runs[a][key]["total"] - runs[b][key]["total"]) < DEGENERATE_NATS]
    return {"verdict": names[order[0]], "tied": tied, "margin_nats": margin,
            "degenerate": degenerate, "order": [names[j] for j in order],
            "p": {c: float(p[j]) for j, c in enumerate(names)},
            "loglik": {c: float(totals[j]) for j, c in enumerate(names)},
            "onset": {c: runs[c][key]["onset"] for c in names}}


def run_mmae(sim, probe_meas=None, ladder_h=LADDER_H, n_frames=None,
             t_onset=None):
    """onset -> bank -> posterior -> verdict, both scorings from one set of replays.

    `probe_meas` supplies NOISY probe readings; without it the probe likelihood
    reads the noiseless truth. `n_frames` truncates the run, which is how Part B
    asks what the verdict would have been on less evidence.

    `t_onset` overrides the onset estimate. The bank is fired BY a detector, and
    the two systems evaluated here run different ones — E walks back the ethanol
    CUSUM arm, E+P walks back whichever of the four channels actually crossed —
    so the onset guess belongs to the caller. Falls back to the ethanol arms
    recorded inside `sim` when it is not given.
    """
    if t_onset is None and sim.alarm_idx is None:
        return {"state": "no_alarm"}
    n_frames = len(sim.rows) if n_frames is None else min(n_frames, len(sim.rows))
    grid = [r["time_h"] for r in sim.rows[:n_frames]]
    meas = [m for m in sim.meas_rows if m["frame"] < n_frames]
    n = len(meas)
    if n == 0:
        return {"state": "no_alarm"}
    meas_by_frame = {m["frame"]: {"z": m["z"], "k": k} for k, m in enumerate(meas)}

    ek = sim.params["ekf"]
    probes = {"sigma": {c: ek[f"sigma_{c}_probe"] for c in ("T", "pH", "DO")}}
    for c in ("T", "pH", "DO"):
        probes[c] = np.array([(probe_meas[c][k] if probe_meas is not None
                               else sim.rows[m["frame"]][c])
                              for k, m in enumerate(meas)])

    t0_hat = (float(t_onset) if t_onset is not None
              else onset_estimate(sim.meas_rows, sim.alarm_idx))
    # Ladder onsets, snapped DOWN to a frame boundary so a hypothesis forks from
    # the shared healthy prefix exactly rather than nearly. Rungs that land on
    # the same frame collapse.
    onsets, frames = [], []
    for off in ladder_h:
        t = max(0.0, t0_hat + off)
        fi = min(max(int(np.searchsorted(grid, t - 1e-12)), 0), n_frames - 1)
        if fi not in frames:
            frames.append(fi)
            onsets.append(grid[fi - 1] if fi > 0 else 0.0)

    # The null hypothesis IS the shared prefix: every faulted hypothesis is
    # identical to it until its own onset, so one pass produces both the null's
    # score and the fork points for all the others.
    null = _Hyp(sim, [])
    ll_null = {"E": np.zeros(n), "P": np.zeros(n)}
    tr_null = _new_trace(n)
    snaps = {fi: {} for fi in frames}
    _replay(sim, null, grid, 0, n_frames, meas_by_frame, ll_null, tr_null,
            probes, snap_at=snaps)

    def _pack(label, onset, ll):
        e, ep = ll["E"], ll["E"] + ll["P"]
        return {"label": label,
                "E":  {"onset": onset, "ll": e,  "total": float(e.sum())},
                "EP": {"onset": onset, "ll": ep, "total": float(ep.sum())}}

    runs = {NULL: _pack("No fault (null)", None, ll_null)}
    for cid, meta in candidate_causes().items():
        best = None
        for fi, t_on in zip(frames, onsets):
            hyp = _Hyp(sim, [normalize_fault({"fault": cid, **meta["params"],
                                              "t_fault": t_on})])
            hyp.restore(snaps[fi])
            ll = {k: v.copy() for k, v in ll_null.items()}   # pre-onset = null
            tr = {k: v.copy() for k, v in tr_null.items()}
            _replay(sim, hyp, grid, fi, n_frames, meas_by_frame, ll, tr, probes)
            cand = _pack(meta["label"], t_on, ll)
            # PROFILE over the onset axis (max), never marginalise (sum).
            # Summing rewards SLOW faults, which degrade gracefully as t0 moves
            # and so accumulate mass across many rungs whatever is true.
            if best is None:
                best = cand
            else:
                for key in ("E", "EP"):
                    if cand[key]["total"] > best[key]["total"]:
                        best[key] = cand[key]
        runs[cid] = best

    names = list(runs)
    return {"state": "ok", "t_onset": t0_hat,
            "scores": {"ethanol": _score(names, runs, "E"),
                       "probes":  _score(names, runs, "EP")}}

## A.9 · The scenario set

Eight rungs: four actuator loops × two failure modes, at the severities the MMAE bank
itself commits to. The injector offers the heater and oxygen loops as *one* entry with a
`%`-left field, because "dead" and "running-but-wrong" are the same code path there — but
both rungs are still injectable, and the bank still carries both as separate hypotheses,
so both are measured here.

Every run: fault at **t = 0.5 h** into a 10 h batch, `robust` organism, ethanol every 5 min.
That onset is the app's, not a choice made here: `backend/test_mmae_isolation.py` injects
every one of its cases at `t_fault = 0.5`. It matters more than it looks — C.2 measures
how far isolation degrades when the same fault lands later in the batch.

In [78]:
T_FAULT = 0.5          # h — the app's own onset (backend/test_mmae_isolation.py), and the
                       # same for every scenario, so the delays below are comparable
CH = ("T", "pH", "DO") # the three probe channels

SCENARIOS = {
    "heater_off":         ({"Q_pct": 0.0},          "Heater dead"),
    "heater_clog":        ({"Q_pct": 6.0},          "Heater underpowered (6 % left)"),
    "acid_pump_off":      ({},                      "Acid pump dead"),
    "acid_pump_stuck":    ({"F_A_stuck": 210.0},    "Acid pump stuck ON"),
    "base_pump_off":      ({},                      "Base pump dead"),
    "base_pump_stuck":    ({"F_B_stuck": 210.0},    "Base pump stuck ON"),
    "agitation_aer_stop": ({"kla_pct": 0.0},        "Agitation/aeration stopped"),
    "agitation_aer_clog": ({"kla_pct": 3.0},        "Agitation/aeration restricted (3 % kLa)"),
}
LABEL = {k: v[1] for k, v in SCENARIOS.items()}
print(len(SCENARIOS), "scenarios, fault at t =", T_FAULT, "h")

8 scenarios, fault at t = 0.5 h


## A.10 · Running one batch, and reading the probes off it

In [79]:
def run_batch(fault_id=None, seed=0, t_fault=T_FAULT, cfg=None):
    """One batch. Returns (sim, P_diag_per_frame, shadow_state_per_frame)."""
    sim = Simulation(cfg)
    sim.rng = np.random.default_rng(seed)          # ethanol sensor noise stream
    if fault_id is not None:
        sim.inject_fault({"fault": fault_id, "t_fault": t_fault,
                          **SCENARIOS[fault_id][0]})
    Pd, xh = [], []
    while not sim.done:
        sim.step()
        Pd.append(np.diag(sim.ekf.P).copy())
        xh.append(sim.ekf.x.copy())
    return sim, np.array(Pd), np.array(xh)


def probe_channels(sim, Pd, xh, seed=0):
    """Noisy probe readings at the ethanol sample times, and their normalised
    innovations against the healthy shadow.

        nu_c = z_c - x_hat_c        S_c = P_cc + sigma_c^2        z = nu_c / sqrt(S_c)

    This is the Design-A logic verbatim, just on three more channels: the shadow
    never sees the probes, so the residual IS the fault signal.
    """
    rng = np.random.default_rng(10_000 + seed)
    sig = {c: sim.params["ekf"][f"sigma_{c}_probe"] for c in CH}
    z = {c: [] for c in CH}
    meas = {c: [] for c in CH}
    for m in sim.meas_rows:
        fr = m["frame"]
        for c in CH:
            zc = sim.rows[fr][c] + rng.normal(0.0, sig[c])
            S = Pd[fr][IDX[c]] + sig[c] ** 2
            meas[c].append(zc)
            z[c].append((zc - xh[fr][IDX[c]]) / np.sqrt(S))
    return ({c: np.array(v) for c, v in z.items()},
            {c: np.array(v) for c, v in meas.items()})


def ethanol_z(sim):
    """The normalised ethanol innovation the CUSUM actually consumes."""
    return np.array([m["nu"] / np.sqrt(m["S"]) for m in sim.meas_rows])


def sample_times(sim):
    return np.array([m["time_h"] for m in sim.meas_rows])


t0 = time.time()
_sim, _Pd, _xh = run_batch(None, seed=0)
print(f"healthy batch: {len(_sim.rows)} frames, {len(_sim.meas_rows)} ethanol samples, "
      f"{time.time()-t0:.1f} s")
print("alarm on a healthy batch:", _sim.alarm_t)

healthy batch: 600 frames, 120 ethanol samples, 1.1 s
alarm on a healthy batch: None


---
# Part B — The detector, and one scenario end to end

In [80]:
# ── B.1 · The detector: a bank of independent two-sided CUSUMs ───────────────
K_PRB, H_PRB = 1.0, 10.0        # the sigma-equivalent of the deployed ethanol pair

SYSTEMS = {
    "ethanol": ("E",),          # E   — the soft sensor alone
    "probes":  ("E",) + CH,     # E+P — the soft sensor AND the rig's own instruments
}
SYS_LABEL = {"ethanol": "E · ethanol only", "probes": "E+P · ethanol + T/pH/DO"}


def cusum_bank(z, channels):
    """Independent two-sided CUSUMs, one per channel. Returns (alarm, arms).

    `alarm` is (sample index, channel) of the first arm to cross, or None.
    `arms` keeps the full (n, 2) trace per channel, because the onset walk-back
    needs the HISTORY of whichever arm fired, not merely the fact that it fired.
    """
    n = len(z["E"])
    arms = {c: np.zeros((n, 2)) for c in channels}
    state = {c: (0.0, 0.0) for c in channels}
    alarm = None
    for i in range(n):
        for c in channels:
            k, h = (K_SLACK, H_CUSUM) if c == "E" else (K_PRB, H_PRB)
            sp, sn, fired = cusum_step(state[c][0], state[c][1], z[c][i], k, h)
            state[c] = (sp, sn)
            arms[c][i] = (sp, sn)
            if fired and alarm is None:
                alarm = (i, c)
    return alarm, arms


def onset_from_arms(arm, idx, times):
    """Last sample time at which the FIRING arm was zero before the alarm.

    The same walk-back as `onset_estimate`, generalised off the ethanol arms
    stored inside `sim` onto whichever channel raised *this system's* alarm. The
    onset guess belongs to the detector, so each system owns its own — and the
    ladder in `run_mmae` is what absorbs the error either way.
    """
    key = 1 if arm[idx, 1] >= arm[idx, 0] else 0
    j = idx
    while j > 0 and arm[j - 1, key] > 0.0:
        j -= 1
    return float(times[j])

In [81]:
# The ethanol arm of this bank must BE the detector that runs inside `Simulation`
# — otherwise system E is a re-implementation being compared against the app's
# real one, and any gap it shows would be an artefact of the rewrite.
_s, _Pd, _xh = run_batch("heater_off", seed=0)
_alarm, _arms = cusum_bank({"E": ethanol_z(_s)}, ("E",))
_i, _ = _alarm
_sp = np.array([m["Sp"] for m in _s.meas_rows])
_sn = np.array([m["Sn"] for m in _s.meas_rows])
print(f"alarm sample   Simulation {_s.alarm_idx}   cusum_bank {_i}"
      f"   identical: {_i == _s.alarm_idx}")
print(f"arm traces     max |delta S+| = {np.abs(_arms['E'][:, 0] - _sp).max():.1e}"
      f"   max |delta S-| = {np.abs(_arms['E'][:, 1] - _sn).max():.1e}")
print(f"onset estimate {onset_estimate(_s.meas_rows, _s.alarm_idx):.4f} h  vs  "
      f"{onset_from_arms(_arms['E'], _i, sample_times(_s)):.4f} h")

alarm sample   Simulation 13   cusum_bank 13   identical: True
arm traces     max |delta S+| = 0.0e+00   max |delta S-| = 0.0e+00
onset estimate 0.7500 h  vs  0.7500 h


## B.2 · One scenario, both systems

`evaluate` runs a batch once and reads it twice. The bank is re-scored over a ladder of
evidence windows measured from *that system's own alarm*, and each window truncates the
record before the bank sees it, so no future information leaks back into the onset choice
or the likelihood.

**Verdict delay is measured from the fault, like the alarm delay**, and it is the first
window from which the verdict is correct *and stays correct to the end of the batch* — a
cause that is named right once and then abandoned has not settled.

The `margin` column is how far the winning cause beat the runner-up, in nats. It is worth
reading beside every verdict: **2.3 nats is 10:1 odds**, the bar below which two causes
are not distinguishable on this data. A correct verdict at 0.2 nats is the right cause
edging ahead, not an identification.

In [82]:
# ── B.2 · one scenario through both systems ─────────────────────────────────
OFFSETS_MIN = [0, 30, 60, 120, 180, 300]   # evidence windows from each system's own alarm


def evaluate(fid, seed=0):
    """One batch, read by both systems. Returns {system: row}."""
    sim, Pd, xh = run_batch(fid, seed=seed)
    zp, pm = probe_channels(sim, Pd, xh, seed=seed)
    z = {"E": ethanol_z(sim), **zp}
    times, grid = sample_times(sim), [r["time_h"] for r in sim.rows]
    out = {}

    for sysname, chans in SYSTEMS.items():
        alarm, arms = cusum_bank(z, chans)
        if alarm is None:
            out[sysname] = {"alarm_min": np.nan, "verdict_min": np.nan,
                            "verdict": "no alarm", "margin": np.nan}
            continue

        i, ch = alarm
        t_alarm = float(times[i])
        t_hat = onset_from_arms(arms[ch], i, times)

        # score each window once, cached by frame count so a window that collapses
        # onto an earlier one is reused rather than re-integrated
        rows, cache = [], {}
        for t in [t_alarm + o / 60.0 for o in OFFSETS_MIN] + [grid[-1]]:
            if t > grid[-1] + 1e-9:
                continue
            n = min(int(np.searchsorted(grid, t - 1e-9)) + 1, len(grid))
            if n not in cache:
                cache[n] = run_mmae(sim, probe_meas=pm, n_frames=n,
                                    t_onset=t_hat)["scores"][sysname]
            s = cache[n]
            rows.append({"t_min": (grid[n - 1] - T_FAULT) * 60.0,   # from the FAULT
                         "verdict": s["verdict"], "ok": s["verdict"] == fid,
                         "margin": s["margin_nats"]})

        tr = pd.DataFrame(rows)
        ok = tr.ok.to_numpy()
        if ok[-1]:                                   # settled: correct from here to the end
            tail = np.flip(np.cumprod(np.flip(ok.astype(int))))
            settled = float(tr.t_min.to_numpy()[np.argmax(tail == 1)])
        else:
            settled = np.nan                         # never settles inside the batch
        out[sysname] = {"alarm_min": (t_alarm - T_FAULT) * 60.0,
                        "verdict_min": settled,
                        "verdict": tr.verdict.iat[-1],
                        "margin": float(tr.margin.iat[-1])}
    return out

---
# Part C — Every scenario, both systems

Eight scenarios: four actuator loops x two failure modes. One batch each.

> This cell integrates the bank once per ladder rung per window, for both systems —
> **about 15 minutes**.

In [83]:
t0 = time.time()
rows = []
for fid in SCENARIOS:
    r = evaluate(fid)
    rows.append({
        "scenario":        LABEL[fid],
        "fault":           fid,
        "E alarm":         r["ethanol"]["alarm_min"],
        "E verdict":       r["ethanol"]["verdict_min"],
        "E names":         r["ethanol"]["verdict"],
        "E margin":        r["ethanol"]["margin"],
        "E+P alarm":       r["probes"]["alarm_min"],
        "E+P verdict":     r["probes"]["verdict_min"],
        "E+P names":       r["probes"]["verdict"],
        "E+P margin":      r["probes"]["margin"],
    })
    print(f"{fid:22s} done  [{time.time()-t0:5.0f} s]")

table = pd.DataFrame(rows).set_index("scenario")
print(f"\n{time.time()-t0:.0f} s total\n")
table.drop(columns="fault").round(1)

heater_off             done  [   52 s]
heater_clog            done  [  105 s]
acid_pump_off          done  [  107 s]
acid_pump_stuck        done  [  159 s]
base_pump_off          done  [  188 s]
base_pump_stuck        done  [  233 s]
agitation_aer_stop     done  [  308 s]
agitation_aer_clog     done  [  388 s]

388 s total



,E alarm,E verdict,E names,E margin,E+P alarm,E+P verdict,E+P names,E+P margin
scenario,,,,,,,,
Heater dead,40.0,160.0,heater_off,0.2,5.0,5.0,heater_off,7538.2
Heater underpowered (6 % left),45.0,45.0,heater_clog,0.3,5.0,35.0,heater_clog,6939.8
Acid pump dead,NaN,NaN,no alarm,NaN,NaN,NaN,no alarm,NaN
Acid pump stuck ON,40.0,160.0,acid_pump_stuck,2.7,5.0,5.0,acid_pump_stuck,289124.3
Base pump dead,NaN,NaN,no alarm,NaN,40.0,40.0,base_pump_off,28573.4
Base pump stuck ON,35.0,335.0,base_pump_stuck,0.0,5.0,5.0,base_pump_stuck,187722.9
Agitation/aeration stopped,85.0,85.0,agitation_aer_stop,105.2,10.0,10.0,agitation_aer_stop,27006.3
Agitation/aeration restricted (3 % kLa),185.0,245.0,agitation_aer_clog,0.3,10.0,40.0,agitation_aer_clog,196531.9


## C.1 · The averages

Averaged over the scenarios each system actually **alarms** on — a fault that never
raises an alarm has no delay to average, and scoring it as a large number instead of as
a miss would flatter the mean. The counts say how many scenarios are behind each figure.

In [84]:
summary = pd.DataFrame({
    sys: {
        "scenarios alarmed":       int(table[f"{sys} alarm"].notna().sum()),
        "verdict correct":         int((table[f"{sys} names"] == table["fault"]).sum()),
        "mean alarm delay [min]":  table[f"{sys} alarm"].mean(),
        "mean verdict delay [min]": table[f"{sys} verdict"].mean(),
    } for sys in ("E", "E+P")
})
print(f"out of {len(SCENARIOS)} scenarios\n")
summary.round(1)

out of 8 scenarios



,E,E+P
scenarios alarmed,6.0,7.0
verdict correct,6.0,7.0
mean alarm delay [min],71.7,11.4
mean verdict delay [min],171.7,20.0


### Reading the table

Two scenarios can never be diagnosed, and both are properties of the **plant**, not of
the estimator:

* **`acid_pump_off` is state-identical to healthy.** A growing batch acidifies, so the PI
  controller only ever calls for base. A pump that is never asked to run cannot fail
  observably — no instrument at any price changes this.
* **`base_pump_off` is silent on ethanol** at this pH window: the batch bottoms out at
  pH 4.47, still ~99 % of peak growth rate, so there is nothing for the soft sensor to
  see. The probes do detect it — that gain is only available because they are allowed
  into the detector as well as the bank.

Read the `margin` columns beside the verdicts. Where ethanol is right, it is usually
right by a few tenths of a nat against a 2.3-nat bar; the probe verdicts win by orders of
magnitude more. "Correct" and "identified" are not the same claim.